# Qualitative comparison — ESDC gap-filling adaptation levels

Displays gap-filling reconstructions on randomly drawn samples from the ESDC
gap-filling evaluation dataset, side by side, for all four adaptation levels
(zero-shot, fine-tuning, structured masking, from-scratch training) plus the
two non-learned interpolation baselines (Ordinary Kriging, linear
interpolation). Each sample's dataset row index is shown so it can be found
again.

Workflow:
1. Run the **imports**, **config**, and **load dataset + models** cells once.
2. Re-run the **browse** cell repeatedly (changing `seed`, or setting
   `sample_indices` directly) to page through random draws.
3. Use the **save** cell to export a page you like as a PNG.

In [ ]:
import os
import sys

import cmocean
import h5py
import numpy as np
import torch
from joblib import Parallel, delayed
from matplotlib import pyplot as plt
from pykrige.ok import OrdinaryKriging
from scipy.interpolate import griddata
from scipy.spatial import QhullError

_gap_aware_root = os.environ.get(
    "GAP_AWARE_ROOT",
    os.path.abspath(os.path.join(os.path.dirname("__file__"), "..", "..", "..")),
)
if _gap_aware_root not in sys.path:
    sys.path.insert(0, _gap_aware_root)

from model_gap_aware.maskfeat import MaskFeatGapFiller
from esdc.src.evaluation.evaluate_gapfilling import (
    ESDCGapFillingEvalDataset,
    find_checkpoint,
    _load_norm_range,
)

In [ ]:
# --- Dataset ---
h5_path = os.path.join(
    os.environ.get(
        "ESDC_GAP_FILLING_DIR",
        os.path.join(_gap_aware_root, "esdc", "data", "lst_gap_filling"),
    ),
    "dataset.h5",
)
device = "cpu"

# --- Checkpoint discovery ---
# One log_dir per adaptation level, matching each exp0N/config.yaml's training.log_dir.
LOG_DIRS = {
    "zero_shot": os.path.join(_gap_aware_root, "esdc/lightning_logs/experiments/exp01_zero_shot"),
    "finetuning": os.path.join(_gap_aware_root, "esdc/lightning_logs/experiments/exp02_finetuning"),
    "structured_masking": os.path.join(_gap_aware_root, "esdc/lightning_logs/experiments/exp03_structured_masking"),
    "from_scratch": os.path.join(_gap_aware_root, "esdc/lightning_logs/experiments/exp04_scratch_gapfilling"),
}

# Manual override: set a checkpoint path for a level to skip auto-discovery
# for it; leave None to auto-discover the lowest-MAE_val checkpoint under
# that level's log_dir via find_checkpoint() (same logic evaluate.sh uses).
MANUAL_CHECKPOINTS = {
    "zero_shot": os.path.join(_gap_aware_root, "esdc/lightning_logs/experiments/exp01_zero_shot/version_1/checkpoints/lst-gapfiller-epoch=48-MAE_val=0.0096.ckpt"),
    "finetuning": os.path.join(_gap_aware_root,"esdc/lightning_logs/experiments/exp02_finetuning/sample_ablation/n0100/version_0/checkpoints/lst-gapfiller-epoch=04-MAE_val=0.0102.ckpt"),
    "structured_masking": os.path.join(_gap_aware_root,"esdc/lightning_logs/experiments/exp03_structured_masking/version_2/checkpoints/lst-gapfiller-epoch=47-MAE_val=0.0127.ckpt"),
    "from_scratch": os.path.join(_gap_aware_root,"esdc/lightning_logs/experiments/exp04_scratch_gapfilling/version_1/checkpoints/lst-gapfiller-epoch=43-MAE_val=0.0142.ckpt"),
}

# MANUAL_CHECKPOINTS = {
#     "zero_shot": None,
#     "finetuning": None,
#     "structured_masking": None,
#     "from_scratch": None,
# }

# --- Interpolation baselines ---
variogram_model = "linear"   # pykrige variogram model, matches interpolation_baseline/config.yaml
n_jobs = 1                   # joblib workers for kriging/linear fits (one fit per displayed sample)

# --- LST display / denormalization ---
NORM_RANGE = _load_norm_range(os.path.dirname(h5_path)) or (180.0, 360.0)
LST_MIN, LST_MAX = NORM_RANGE
SCALE_K = LST_MAX - LST_MIN


def denorm_lst(arr):
    return arr * (LST_MAX - LST_MIN) + LST_MIN

In [ ]:
dataset = ESDCGapFillingEvalDataset(h5_path)
print(f"Dataset size: {len(dataset)} samples")

models = {}
for level, log_dir in LOG_DIRS.items():
    ckpt_path = find_checkpoint(log_dir, MANUAL_CHECKPOINTS[level])
    print(f"[{level}] {ckpt_path}")
    m = MaskFeatGapFiller.load_from_checkpoint(ckpt_path, map_location=device)
    m.to(device)
    m.eval()
    models[level] = m

In [ ]:
def krige_predict(x, gt_middle, gap_mask, variogram_model):
    """Mirrors evaluate_kriging_baseline.py::_krige_sample's fit/predict, but
    returns the full reconstructed (H, W) frame instead of an aggregated error."""
    lst_mid = x[2, 0].numpy()
    gt = gt_middle[0].numpy()
    mask = gap_mask.numpy()

    obs_rows, obs_cols = np.where(~np.isnan(lst_mid))
    gap_rows, gap_cols = np.where((mask > 0) & ~np.isnan(gt))

    pred_full = lst_mid.copy()
    if len(gap_rows) == 0 or len(obs_rows) < 3:
        return pred_full

    ok = OrdinaryKriging(
        obs_cols.astype(float), obs_rows.astype(float), lst_mid[obs_rows, obs_cols],
        variogram_model=variogram_model, verbose=False, enable_plotting=False,
    )
    pred, _ = ok.execute("points", gap_cols.astype(float), gap_rows.astype(float))
    pred_full[gap_rows, gap_cols] = np.asarray(pred)
    return pred_full


def linear_interp_predict(x, gt_middle, gap_mask):
    """Mirrors evaluate_linear_interpolation_baseline.py::_linear_interp_sample's
    fit/predict, but returns the full reconstructed (H, W) frame."""
    lst_mid = x[2, 0].numpy()
    gt = gt_middle[0].numpy()
    mask = gap_mask.numpy()

    obs_rows, obs_cols = np.where(~np.isnan(lst_mid))
    gap_rows, gap_cols = np.where((mask > 0) & ~np.isnan(gt))

    pred_full = lst_mid.copy()
    if len(gap_rows) == 0 or len(obs_rows) < 3:
        return pred_full

    obs_points = np.column_stack((obs_cols, obs_rows)).astype(float)
    obs_values = lst_mid[obs_rows, obs_cols]
    gap_points = np.column_stack((gap_cols, gap_rows)).astype(float)

    try:
        pred = griddata(obs_points, obs_values, gap_points, method="linear")
    except QhullError:
        pred = np.full(len(gap_points), np.nan)

    nan_mask = np.isnan(pred)
    if nan_mask.any():
        pred[nan_mask] = griddata(obs_points, obs_values, gap_points[nan_mask], method="nearest")

    pred_full[gap_rows, gap_cols] = pred
    return pred_full


def run_nn_methods(idx):
    """Run all four adaptation-level models on dataset[idx]. Returns a dict
    with the sample's normalized fields and per-method predictions/MAE
    (in normalized [0, 1] units, over gap pixels with valid ground truth)."""
    x, gt_middle, gap_mask = dataset[idx]
    lst_mid_norm = x[2, 0].numpy()
    gt_gap_norm = gt_middle[0].numpy()
    gap_mask_np = gap_mask.numpy() > 0.5
    gt_full_norm = np.where(gap_mask_np, gt_gap_norm, lst_mid_norm).astype(np.float32)
    valid = gap_mask_np & ~np.isnan(gt_gap_norm)

    preds, maes = {}, {}
    x_in = x.unsqueeze(0).to(device)
    with torch.no_grad():
        for level, model in models.items():
            pred, _ = model(x_in, gap_only_masking=True)
            pred_lst = pred[0, 0].cpu().numpy()
            preds[level] = np.where(gap_mask_np, pred_lst, lst_mid_norm)
            maes[level] = (
                float(np.abs(pred_lst[valid] - gt_gap_norm[valid]).mean())
                if valid.any() else float("nan")
            )

    return dict(
        idx=idx, x=x, gt_middle=gt_middle, gap_mask=gap_mask,
        lst_mid_norm=lst_mid_norm, gt_full_norm=gt_full_norm,
        gap_mask_np=gap_mask_np, valid=valid, preds=preds, maes=maes,
    )


def add_baseline_predictions(results_list, variogram_model, n_jobs):
    """Fits Kriging + linear interpolation for each sample in results_list
    (in place) and adds their predictions/MAE alongside the NN methods'."""

    def _fit_one(r):
        kriging_full = krige_predict(r["x"], r["gt_middle"], r["gap_mask"], variogram_model)
        linear_full = linear_interp_predict(r["x"], r["gt_middle"], r["gap_mask"])
        return kriging_full, linear_full

    fitted = Parallel(n_jobs=n_jobs)(delayed(_fit_one)(r) for r in results_list)

    for r, (kriging_full, linear_full) in zip(results_list, fitted):
        gt_gap_norm = r["gt_middle"][0].numpy()
        valid = r["valid"]
        for name, pred_full in (("kriging", kriging_full), ("linear", linear_full)):
            r["preds"][name] = pred_full
            r["maes"][name] = (
                float(np.abs(pred_full[valid] - gt_gap_norm[valid]).mean())
                if valid.any() else float("nan")
            )
    return results_list


# --- Sample filtering by LST temperature band --------------------------------
KELVIN_0C = 273.15
_eligible_idx_cache = {}


def eligible_sample_indices(min_celsius=None, max_celsius=None):
    """Dataset row indices whose clean middle-frame LST lies entirely within
    [min_celsius, max_celsius] degrees Celsius (either bound may be None for
    open-ended). Used to constrain the random fill draw in the browse cell --
    pinned `sample_indices` are always shown and bypass this filter.

    Results are cached per (min, max) so re-running the browse cell with a new
    seed does not rescan the HDF5.
    """
    key = (min_celsius, max_celsius)
    if key not in _eligible_idx_cache:
        if min_celsius is None and max_celsius is None:
            _eligible_idx_cache[key] = np.arange(len(dataset))
        else:
            with h5py.File(h5_path, "r") as f:
                x_mid = f["X_train"][:, MIDDLE_T, :, :, 0]   # (N, H, W) normalized
                y_gap = f["y_train"][:, 0, :, :, 0]          # (N, H, W), NaN off-gap
            # full clean middle frame = observed pixels + gap-pixel ground truth
            clean_c = denorm_lst(np.where(np.isnan(y_gap), x_mid, y_gap)) - KELVIN_0C
            lo = -np.inf if min_celsius is None else float(min_celsius)
            hi = np.inf if max_celsius is None else float(max_celsius)
            with np.errstate(invalid="ignore"):
                vmin = np.nanmin(clean_c, axis=(1, 2))
                vmax = np.nanmax(clean_c, axis=(1, 2))
            ok = np.isfinite(vmin) & (vmin >= lo) & (vmax <= hi)
            _eligible_idx_cache[key] = np.flatnonzero(ok)
    return _eligible_idx_cache[key]


ALL_METHODS = ["zero_shot", "finetuning", "structured_masking", "from_scratch", "kriging", "linear"]
DISPLAY_NAMES = {
    "zero_shot": "Zero-shot",
    "finetuning": "Fine-tuning",
    "structured_masking": "Structured masking",
    "from_scratch": "From-scratch",
    "kriging": "Kriging",
    "linear": "Linear interp.",
}

N_TIMESTEPS = 5
MIDDLE_T = N_TIMESTEPS // 2  # 2

TIMESTEP_TITLES = [f"t={t}" if t != MIDDLE_T else f"t={t} (gappy)" for t in range(N_TIMESTEPS)]
PRED_ROW_TITLES = ["Ground truth"] + [DISPLAY_NAMES[m] for m in ALL_METHODS]

# Column layout for the single-row-per-sample figure:
#   [N_TIMESTEPS input frames] [spacer] [ground truth] [spacer] [len(ALL_METHODS) results]
# The spacer columns hold NO axes -- their width alone sets the gap between the
# three groups. PANEL_WSPACE sets the (smaller) gap between adjacent panels.
SPACER_RATIO = 0.06   # width of each blank spacer column, relative to an image column
PANEL_WSPACE = 0.035  # gap between adjacent image panels, as a fraction of panel width
COL_WIDTH_RATIOS = (
    [1.0] * N_TIMESTEPS
    + [SPACER_RATIO, 1.0, SPACER_RATIO]
    + [1.0] * len(ALL_METHODS)
)
N_COLS = len(COL_WIDTH_RATIOS)                 # 14
GT_COL = N_TIMESTEPS + 1                       # 6
FIRST_METHOD_COL = N_TIMESTEPS + 3             # 8

CMAP = cmocean.cm.matter_r.copy()
VMIN, VMAX = 243.15, 303.15   # display range in Kelvin
masked_cmap = CMAP.copy()
# masked_cmap.set_bad("lightgray")

## Browse

Re-run this cell to page through new random draws (change `seed`), or pin
specific dataset row indices via `sample_indices` (e.g. `sample_indices =
[1234, 5821]`). The page always shows `n_samples` rows: pinned indices come
first, and any remaining rows are filled with random draws. Set
`sample_indices = None` for a fully random page.

`lst_filter_min_c` / `lst_filter_max_c` restrict the **random fill** to samples
whose clean middle-frame LST lies entirely within that Celsius band (either
bound may be `None`). Pinned `sample_indices` are always shown regardless of
the filter. The eligible-index scan is cached per band.

In [ ]:
seed = None            # change this to get a new random draw
n_samples = 6        # samples per page -- the page is always filled to this many
sample_indices = [2772, 3554, 385]  # specific IDs to pin; any remaining slots are filled at random

# Restrict the random fill draw to samples whose clean middle-frame LST lies
# entirely within this Celsius band (None = open-ended on that side). The pinned
# `sample_indices` above are always shown and are NOT subject to this filter.
lst_filter_min_c = 0   # e.g. -10
lst_filter_max_c = 30   # e.g. 30

rng = np.random.default_rng(seed)

requested = [] if sample_indices is None else list(dict.fromkeys(sample_indices))
n_fill = max(0, n_samples - len(requested))
if n_fill:
    pool = np.setdiff1d(
        eligible_sample_indices(lst_filter_min_c, lst_filter_max_c), requested
    )
    if len(pool) < n_fill:
        print(
            f"Warning: only {len(pool)} sample(s) match the LST filter "
            f"[{lst_filter_min_c}, {lst_filter_max_c}] °C -- showing all of them."
        )
    fill = rng.choice(pool, size=min(n_fill, len(pool)), replace=False)
    sample_indices = requested + sorted(fill.tolist())
else:
    sample_indices = requested

print(f"Showing {len(sample_indices)} samples: {sample_indices}")

results = [run_nn_methods(idx) for idx in sample_indices]
results = add_baseline_predictions(results, variogram_model, n_jobs)

# One row per sample: [5 input frames] | spacer | [ground truth] | spacer | [6 results].
# Axes are created only in the real columns; the two spacer columns stay empty so
# their width alone drives the larger inter-group gap.
fig = plt.figure(
    figsize=(2.6 * (N_TIMESTEPS + 1 + len(ALL_METHODS)) + 1.0, 2.9 * len(results)),
    layout="constrained",
)
fig.get_layout_engine().set(w_pad=0.02, h_pad=0.03, wspace=PANEL_WSPACE, hspace=0.03)
gs = fig.add_gridspec(len(results), N_COLS, width_ratios=COL_WIDTH_RATIOS)

for i, r in enumerate(results):
    gt_disp = denorm_lst(r["gt_full_norm"])

    # --- input frames (first N_TIMESTEPS columns) ---
    for t in range(N_TIMESTEPS):
        ax = fig.add_subplot(gs[i, t])
        frame_disp = denorm_lst(r["x"][t, 0].numpy())
        ax.imshow(np.ma.masked_invalid(frame_disp), cmap=masked_cmap, vmin=VMIN, vmax=VMAX)
        ax.set_xticks([])
        ax.set_yticks([])
        if t == 0:
            ax.set_ylabel(f"idx={r['idx']}", fontsize=11, fontweight="bold")
        if i == 0:
            ax.set_title(TIMESTEP_TITLES[t])

    # --- ground truth ---
    ax = fig.add_subplot(gs[i, GT_COL])
    ax.imshow(gt_disp, cmap=CMAP, vmin=VMIN, vmax=VMAX)
    ax.set_xticks([])
    ax.set_yticks([])
    if i == 0:
        ax.set_title(PRED_ROW_TITLES[0])

    # --- gap-filling results ---
    for j, method in enumerate(ALL_METHODS):
        ax = fig.add_subplot(gs[i, FIRST_METHOD_COL + j])
        pred_disp = denorm_lst(r["preds"][method])
        ax.imshow(np.ma.masked_invalid(pred_disp), cmap=CMAP, vmin=VMIN, vmax=VMAX)
        ax.set_xticks([])
        ax.set_yticks([])
        mae = r["maes"][method]
        mae_str = f"MAE={mae * SCALE_K:.2f} K" if not np.isnan(mae) else "MAE=n/a"
        ax.set_title(
            f"{PRED_ROW_TITLES[1 + j]}\n{mae_str}" if i == 0 else mae_str,
            fontsize=9,
        )

fig.suptitle(
    "ESDC gap-filling: qualitative comparison across adaptation levels",
    fontsize=13,
)
plt.show()

## Save

Export the current page's figure as a PNG once you've found a good set of
samples.

In [ ]:
def save_current_figure(fig, sample_indices, out_dir=None):
    if out_dir is None:
        out_dir = os.path.join(_gap_aware_root, "esdc", "experiments", "qualitative", "figures")
    os.makedirs(out_dir, exist_ok=True)
    fname = "sample_" + "_".join(str(i) for i in sample_indices) + ".png"
    out_path = os.path.join(out_dir, fname)
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    print(f"Saved: {out_path}")
    return out_path


save_current_figure(fig, sample_indices)